# IFSP — Câmpus Presidente Epitácio
## Bacharelado em Ciência da Computação • Aprendizado de Máquina (PEPAPRM)
### Atividade Prática: Três Bases, Três Problemas
---
# Base 1: Previsão de Cancelamento de Clientes (Churn)
**Pergunta de Negócio:** *Quem vai cancelar o serviço de internet nos próximos 6 meses?*


## 1. Identificação do Problema e Justificativa

Aplicando o fluxograma **Tenho um TARGET ($y$)?**:
- A base possui a coluna alvo `cancelou`, que assume valores discretos e categóricos ($0 =$ permaneceu, $1 =$ cancelou o serviço).
- Portanto, trata-se de um problema de **Aprendizado Supervisionado de Classificação Binária**.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    roc_auc_score,
    RocCurveDisplay
)

# Fixando semente global para reprodutibilidade
RANDOM_STATE = 42


## 2. Carregamento e Inspeção Inicial dos Dados


In [ ]:
df = pd.read_csv('base1_classificacao_churn.csv')
print(f"Dimensões do dataset original: {df.shape[0]} linhas x {df.shape[1]} colunas")
df.head()


In [ ]:
df.info()


## 3. Perguntas de Análise e Tratamento de Dados

### Pergunta 1: Qual é a proporção de cada classe? Um modelo que sempre responde "não cancela" seria bom? Que métrica você usará e por quê?
- **Proporção**: A base apresenta desbalanceamento moderado (~78% classe 0 e ~22% classe 1).
- **Modelo ingênuo**: Um modelo ingênuo que sempre preveja a classe majoritária (0) alcançará ~78% de acurácia aparente, mas terá **Recall de 0% na classe 1**, falhando completamente no objetivo de negócio (identificar clientes em risco).
- **Métrica**: A acurácia é inadequada. Utilizaremos **F1-Score da classe 1 (churn)** e **ROC-AUC** como métricas principais, pois equilibram Precision e Recall e mensuram a capacidade discriminatória real do classificador.


In [ ]:
proporcao = df['cancelou'].value_counts(normalize=True) * 100
print("Distribuição da variável alvo (cancelou):")
for classe, perc in proporcao.items():
    print(f"  Classe {classe} ({'Cancelou' if classe==1 else 'Permaneceu'}): {perc:.2f}%")


### Pergunta 2: Há valores ausentes, linhas repetidas ou colunas que não deveriam entrar no modelo? O que você fez com cada uma?
1. **Linhas repetidas**: Identificadas e removidas com `drop_duplicates()`.
2. **Coluna não preditiva**: `id_cliente` é um identificador arbitrário sem poder causal e causaria sobreajuste. Ela é descartada da matriz de atributos $X$.
3. **Valores ausentes**:
   - `idade`: 25 nulos $\rightarrow$ imputados pela **mediana** dentro do Pipeline.
   - `uso_dados_gb`: 40 nulos $\rightarrow$ imputados pela **mediana** dentro do Pipeline.
   - `forma_pagamento`: 12 nulos $\rightarrow$ imputados pela **moda** (mais frequente) dentro do Pipeline.


In [ ]:
# 1. Linhas repetidas
duplicadas = df.duplicated().sum()
print(f"Linhas duplicadas encontradas: {duplicadas}")
df = df.drop_duplicates()

# 2. Valores ausentes
print("\nValores ausentes por coluna:")
print(df.isnull().sum()[df.isnull().sum() > 0])


### Pergunta 3: Quais colunas são numéricas e quais são texto? O que o algoritmo exige de cada tipo?
- **Numéricas**: `idade`, `tempo_cliente_meses`, `mensalidade`, `chamados_suporte_6m`, `uso_dados_gb`, `atrasos_pagamento_12m`. Exigem tratamento de nulos e padronização (`StandardScaler`) para garantir que variáveis de grandezas distintas não distorçam os coeficientes.
- **Texto/Categóricas**: `plano`, `forma_pagamento`. Algoritmos de aprendizado de máquina exigem tensores numéricos; portanto, aplicamos One-Hot Encoding (`OneHotEncoder(drop='first')`).


In [ ]:
# Separação de X e y
X = df.drop(columns=['id_cliente', 'cancelou'])
y = df['cancelou']

colunas_num = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
colunas_cat = X.select_dtypes(include=['object', 'str']).columns.tolist()

print(f"Atributos numéricos ({len(colunas_num)}): {colunas_num}")
print(f"Atributos categóricos ({len(colunas_cat)}): {colunas_cat}")


## 4. Divisão Treino e Teste (Sem Data Leakage)
A divisão é feita antes de qualquer transformação com estratificação (`stratify=y`) para preservar a proporção de cancelamentos.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=RANDOM_STATE
)
print(f"Conjunto de Treino: {X_train.shape[0]} instâncias")
print(f"Conjunto de Teste:  {X_test.shape[0]} instâncias")


## 5. Construção do Pipeline de Pré-Processamento


In [ ]:
pipeline_num = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

pipeline_cat = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', pipeline_num, colunas_num),
    ('cat', pipeline_cat, colunas_cat)
])


## 6. Escolha e Justificativa dos Algoritmos Candidatos (+ Baseline)

1. **DummyClassifier (Baseline)**: Sempre prevê a classe mais frequente, servindo de linha de base mínima.
2. **Logistic Regression (Candidato 1)**: Modelo linear interpretável com ajuste de pesos (`class_weight='balanced'`) para lidar eficazmente com classes desbalanceadas.
3. **Random Forest (Candidato 2)**: Ensemble baseado em árvores capaz de capturar relações não-lineares e interações entre atrasos e suporte.
4. **HistGradientBoosting (Candidato 3)**: Algoritmo de boosting com alto poder preditivo e regularização nativa.


In [ ]:
modelos = {
    'Dummy (Baseline)': DummyClassifier(strategy='most_frequent'),
    'Regressão Logística': LogisticRegression(class_weight='balanced', random_state=RANDOM_STATE, max_iter=1000),
    'Random Forest': RandomForestClassifier(class_weight='balanced', random_state=RANDOM_STATE),
    'HistGradientBoosting': HistGradientBoostingClassifier(random_state=RANDOM_STATE)
}


## 7. Validação Cruzada Estratificada (K-Fold)
Usaremos `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` avaliando com F1-Score da classe de cancelamento e ROC-AUC.


In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

resultados_cv = []
for nome, modelo in modelos.items():
    pipe = Pipeline([
        ('preprocessor', preprocessor),
        ('classificador', modelo)
    ])
    f1_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='f1')
    auc_scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring='roc_auc')
    resultados_cv.append({
        'Modelo': nome,
        'F1-Score CV (Média)': f1_scores.mean(),
        'F1-Score CV (Std)': f1_scores.std(),
        'ROC-AUC CV (Média)': auc_scores.mean(),
        'ROC-AUC CV (Std)': auc_scores.std()
    })

df_cv = pd.DataFrame(resultados_cv)
df_cv


## 8. Otimização de Hiperparâmetros (GridSearchCV)
A Regressão Logística obteve a melhor sensibilidade e equilíbrio de F1 na classe minoritária. Otimizaremos o hiperparâmetro de regularização $C$.


In [ ]:
pipe_opt = Pipeline([
    ('preprocessor', preprocessor),
    ('classificador', LogisticRegression(class_weight='balanced', random_state=RANDOM_STATE, max_iter=1000))
])

param_grid = {
    'classificador__C': [0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0]
}

grid_search = GridSearchCV(
    estimator=pipe_opt,
    param_grid=param_grid,
    cv=cv,
    scoring='f1',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print(f"Melhor parâmetro encontrado: {grid_search.best_params_}")
print(f"Melhor F1-score em validação cruzada: {grid_search.best_score_:.4f}")

melhor_modelo = grid_search.best_estimator_


## 9. Avaliação Final no Conjunto de Teste (Executada Uma Única Vez)


In [ ]:
y_pred = melhor_modelo.predict(X_test)
y_pred_proba = melhor_modelo.predict_proba(X_test)[:, 1]

print("=== RELATÓRIO DE CLASSIFICAÇÃO NO TESTE ===")
print(classification_report(y_test, y_pred, target_names=['Permaneceu (0)', 'Cancelou (1)']))


In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

# Matriz de Confusão
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred,
    display_labels=['Permaneceu', 'Cancelou'],
    cmap='Blues',
    ax=ax[0]
)
ax[0].set_title('Matriz de Confusão (Conjunto de Teste)')

# Curva ROC
RocCurveDisplay.from_predictions(
    y_test, y_pred_proba,
    name='Regressão Logística',
    ax=ax[1]
)
ax[1].plot([0, 1], [0, 1], 'r--', label='Classificador Aleatório')
ax[1].set_title(f'Curva ROC (AUC = {roc_auc_score(y_test, y_pred_proba):.3f})')
ax[1].legend()

plt.tight_layout()
plt.show()


## 10. Conclusão e Resposta de Negócio

### Qual tipo de erro é mais caro para a operadora (deixar um cliente cancelar sem aviso ou fazer oferta a quem ficaria)? O modelo está equilibrado para isso?

- **Análise de Custos**: O **Falso Negativo** (cliente que iria cancelar, mas o modelo não detectou) é o erro **mais caro**, pois o Custo de Aquisição de Clientes (CAC) de um novo assinante de internet é substancialmente superior ao custo de uma ação de retenção (ex: desconto ou bonificação). O **Falso Positivo** (fazer oferta preventiva para quem ficaria) gera apenas um custo operacional marginal.
- **Equilíbrio do Modelo**: Com a configuração balanceada (`class_weight='balanced'`), o modelo alcançou um **Recall de ~59% a 60%** na classe de cancelamento no teste (detectando a maioria dos cancelamentos potenciais), enquanto o baseline Dummy tinha 0% de Recall. O modelo está alinhado à prioridade estratégica de retenção e prevenção de churn.
